# 04. Статистический анализ

## Содержание
1. [Загрузка данных](#загрузка-данных)
2. [Общая статистика](#общая-статистика)
3. [Z-test (проверка значимости)](#z-test-проверка-значимости)
4. [Доверительный интервал](#доверительный-интервал)
5. [Байесовский подход](#байесовский-подход)
6. [Стратификация](#стратификация)
7. [Выводы](#выводы)

## Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.data_loader import load_results_data
from src.stats import z_test_proportions, confidence_interval, bayesian_beta
from src.metrics import absolute_difference, relative_difference
from src.visualizations import (
    plot_daily_conversion, 
    plot_bayesian_posterior,
    plot_stratification,
    plot_confidence_intervals
)

df = load_results_data()
print(f"Загружено {len(df)} строк")

## Общая статистика

In [ ]:
# Разделяем группы
control = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

n_control = len(control)
n_treatment = len(treatment)
clicks_control = control['converted'].sum()
clicks_treatment = treatment['converted'].sum()
cr_control = clicks_control / n_control
cr_treatment = clicks_treatment / n_treatment

print("Общая статистика\n")
print(f"{'Показатель':<20} {'Control':>12} {'Treatment':>12}")
print(f"{'Пользователей':<20} {n_control:>12,} {n_treatment:>12,}")
print(f"{'Кликов':<20} {clicks_control:>12,} {clicks_treatment:>12,}")
print(f"{'Конверсия':<20} {cr_control:>11.2%} {cr_treatment:>11.2%}")

diff_abs = absolute_difference(cr_control, cr_treatment)
diff_rel = relative_difference(cr_control, cr_treatment)

print(f"\n📈 Разница конверсий: {diff_abs:+.2f} п.п.")
print(f"   Относительный прирост: {diff_rel:+.1f}%")

## Z-test (проверка значимости)

In [ ]:
# Z-test
z_result = z_test_proportions(
    successes_a=clicks_control, n_a=n_control,
    successes_b=clicks_treatment, n_b=n_treatment,
    one_sided=True
)

print("Z-test (односторонний)\n")
print(f"Z-статистика: {z_result['z_stat']:.4f}")
print(f"p-value: {z_result['p_value']:.6f}")
print(f"Статус: {'Значимо' if z_result['is_significant'] else 'Не значимо'}")
print(f"Интерпретация: {z_result['significance_text']}")

if z_result['is_significant']:
    print("\np-value < 0.05 → отвергаем H₀")
    print("   Разница конверсий статистически значима")
else:
    print("\np-value >= 0.05 → не отвергаем H₀")
    print("   Разница конверсий не значима")

## Доверительный интервал

In [ ]:
# Доверительный интервал
ci = confidence_interval(z_result['diff'], z_result['se'], ci=95)

print("95% Доверительный интервал\n")
print(f"[{ci['lower']*100:+.2f} п.п.; {ci['upper']*100:+.2f} п.п.]")

if ci['all_positive']:
    print("Доверительный интервал полностью > 0 → эффект положительный")
elif ci['all_negative']:
    print("Доверительный интервал полностью < 0 → эффект отрицательный")
else:
    print("Доверительный интервал пересекает 0 → эффект не определён")

In [ ]:
# Визуализация доверительного интервала
plot_confidence_intervals(cr_control, cr_treatment, z_result['se'], save_path='../reports/images/ci.png')

## Байесовский подход

In [ ]:
# Байесовский анализ
bayes = bayesian_beta(
    control_clicks=clicks_control,
    control_users=n_control,
    treatment_clicks=clicks_treatment,
    treatment_users=n_treatment,
    n_samples=100000
)

print("Байесовская оценка\n")
print(f"Вероятность, что новый дизайн лучше старого: {bayes['prob_treatment_better']:.1%}")
print(f"Ожидаемая потеря при внедрении: {bayes['expected_loss_pct']:.3f} п.п.")
print(f"Медиана Control: {bayes['median_control']:.2%}")
print(f"Медиана Treatment: {bayes['median_treatment']:.2%}")

if bayes['prob_treatment_better'] > 0.95:
    print("Высокая уверенность, что новый дизайн лучше (95%+)")
elif bayes['prob_treatment_better'] > 0.80:
    print("Умеренная уверенность (80-95%)")
else:
    print("Низкая уверенность (<80%)")

In [ ]:
# Визуализация апостериорных распределений
plot_bayesian_posterior(
    clicks_control, n_control,
    clicks_treatment, n_treatment,
    save_path='../reports/images/bayesian_posterior.png'
)

## Стратификация

In [ ]:
# Стратификация по типу пользователя
segmented_stats = df.groupby(['group', 'user_type']).agg(
    users=('user_id', 'count'),
    clicks=('converted', 'sum'),
    cr=('converted', 'mean')
).reset_index()

print("Стратификация по типу пользователя\n")
print(segmented_stats.to_string(index=False))

print("\nЭффект по сегментам:")
for user_type in ['new', 'old']:
    cr_c = segmented_stats[(segmented_stats['group']=='control') & (segmented_stats['user_type']==user_type)]['cr'].values[0]
    cr_t = segmented_stats[(segmented_stats['group']=='treatment') & (segmented_stats['user_type']==user_type)]['cr'].values[0]
    diff = cr_t - cr_c
    rel = (cr_t / cr_c - 1) * 100
    print(f"  {user_type}: +{diff*100:.2f} п.п. (+{rel:.1f}%)")

In [ ]:
# Визуализация стратификации
plot_stratification(df, save_path='../reports/images/stratification.png')

## Выводы

### Результаты статистического анализа

| Показатель | Control | Treatment | Эффект |
|------------|---------|-----------|--------|
| **Пользователей** | 5,521 | 5,521 | — |
| **Конверсия** | 6.38% | 9.67% | +3.30 п.п. (+51.7%) |

### Статистическая значимость

| Тест | Результат | Статус |
|------|-----------|--------|
| **Z-test** (односторонний) | p ≈ 0.000 | ✅ Значимо |
| **95% Доверительный интервал** | [+2.28 п.п.; +4.31 п.п.] | ✅ Полностью > 0 |
| **Байесовская вероятность** | 99.9% | ✅ Высокая уверенность |

### Стратификация

| Тип пользователя | Control | Treatment | Эффект |
|------------------|---------|-----------|--------|
| **Новые** | 5.61% | 9.00% | +3.39 п.п. (+60.3%) |
| **Старые** | 6.90% | 10.14% | +3.24 п.п. (+46.9%) |

### Итог

> **Эффект статистически значим и сохраняется во всех сегментах.**

Достигнутый эффект (+51.7%) значительно превышает планируемый MDE (20%).